# 🧪 W3-D3 概念实验：RLHF 奖励模型与 PPO 训练模拟

> 配套阅读：`ima/第3周-Day3-RLHF人类反馈强化学习详解.md`
>
> RLHF 三步走：SFT→奖励模型→PPO优化。用纯 Python 模拟每一步。

## 实验 1：模拟奖励模型的偏好排序

奖励模型：给好回答高分、差回答低分。

In [ ]:
import numpy as np

class SimpleRewardModel:
    def __init__(self):
        self.good = {'详细':0.3,'具体':0.25,'例如':0.2,'首先':0.15,'步骤':0.2,'总结':0.15,'注意':0.15}
        self.bad = {'不知道':-0.3,'随便':-0.25,'不清楚':-0.2,'无法':-0.2}
    def score(self, answer):
        r = 0.0
        for k, w in self.good.items():
            if k in answer: r += w
        for k, w in self.bad.items():
            if k in answer: r += w
        if 50 < len(answer) < 500: r += 0.3
        if len(answer) < 20: r -= 0.2
        if '。' in answer: r += 0.1
        return round(r, 3)

rm = SimpleRewardModel()
answers = [
    "首先，建议选择Python入门。具体可通过在线教程学习。例如每天练习小项目。注意坚持，总结经验很重要。",
    "不知道，随便学吧。",
    "机器学习是AI的分支。",
    "学习编程步骤：第一步选Python。第二步找教程。第三步多写代码。总结：贵在坚持。",
]
print("奖励模型打分（问题：如何学习编程？）")
for i, a in enumerate(answers):
    print(f"  回答{i+1}: {rm.score(a):+.3f}  →  {a[:25]}…")
print("→ 详细+有结构=高分；敷衍=低分。")

## 实验 2：PPO clip 机制 — 限制每步变化幅度

对比有/无 clip 的训练轨迹。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import font_manager
font_path = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"
font_manager.fontManager.addfont(font_path)
font_name = font_manager.FontProperties(fname=font_path).get_name()
plt.rcParams["font.family"] = font_name
plt.rcParams["axes.unicode_minus"] = False
np.random.seed(2); steps = 80
pol_nc, pol_cl = np.zeros(steps), np.zeros(steps)
p = 0.5
for i in range(steps):
    p += 0.15*(0.02*np.sin(i/5)+0.01) + np.random.normal(0, 0.02)
    pol_nc[i] = np.clip(p, 0, 1)
p = 0.5
for i in range(steps):
    s = np.clip(0.15*(0.02*np.sin(i/5)+0.01), -0.02, 0.02)
    p += s + np.random.normal(0, 0.005)
    pol_cl[i] = np.clip(p, 0, 1)
fig, (a1,a2) = plt.subplots(1, 2, figsize=(12, 4.5))
a1.plot(range(steps), pol_nc, 'r-', lw=1.5); a1.set_title('无 clip：策略剧烈震荡')
a1.set_ylabel('策略参数'); a1.set_xlabel('Step'); a1.grid(True, alpha=0.3)
a2.plot(range(steps), pol_cl, 'b-', lw=1.5); a2.set_title('PPO clip：平稳收敛')
a2.set_ylabel('策略参数'); a2.set_xlabel('Step'); a2.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()
print("→ PPO clip 像「每次只调一点点」的教练。")

## 实验 3：KL 散度惩罚 — 防止偏离 SFT 太远

总奖励 = 奖励 − β × KL。β 太小→作弊，太大→不学习。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import font_manager
font_path = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"
font_manager.fontManager.addfont(font_path)
font_name = font_manager.FontProperties(fname=font_path).get_name()
plt.rcParams["font.family"] = font_name
plt.rcParams["axes.unicode_minus"] = False
np.random.seed(3); steps = 100
betas = [(0.001,'β=0.001（太松）','red'),(0.05,'β=0.05（适中）','green'),(0.5,'β=0.5（太紧）','blue')]
fig, ax = plt.subplots(figsize=(10, 5.5))
for beta, label, color in betas:
    rew, p = [], 0.5
    for i in range(steps):
        kl = (p-0.5)**2; total = min(1.0, p*1.2) - beta*kl
        p += 0.01*total + np.random.normal(0, 0.005); p = np.clip(p, 0, 1)
        rew.append(total + 0.1)
    ax.plot(range(steps), rew, '-', color=color, lw=2, label=label)
ax.set_xlabel('训练步数'); ax.set_ylabel('总奖励')
ax.set_title('KL 惩罚 β 大小决定模型偏离 SFT 多远')
ax.legend(fontsize=10); ax.grid(True, alpha=0.3); plt.tight_layout(); plt.show()
print("β太小→作弊；β适中→稳步提升；β太大→不学习")

## 实验 4：RLHF 三阶段质量跃升

预训练→SFT→RLHF，每一步提升不同维度。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import font_manager
font_path = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"
font_manager.fontManager.addfont(font_path)
font_name = font_manager.FontProperties(fname=font_path).get_name()
plt.rcParams["font.family"] = font_name
plt.rcParams["axes.unicode_minus"] = False
stages = ['预训练\n(海量文本)','SFT\n(指令数据)','RLHF\n(人类偏好)']
quality = [40, 68, 92]; colors = ['#95a5a6','#3498db','#e74c3c']
fig, ax = plt.subplots(figsize=(9, 5))
ax.bar(range(3), quality, color=colors, alpha=0.85, width=0.5, edgecolor='black')
for i in range(2):
    ax.annotate('', xy=(i+0.6, quality[i+1]), xytext=(i+0.4, quality[i]),
                arrowprops=dict(arrowstyle='->', lw=2, color='gray'))
for i,(q,s) in enumerate(zip(quality, stages)):
    ax.text(i, q+2, f'{q}分', ha='center', fontsize=13, fontweight='bold')
    ax.text(i, -7, s, ha='center', fontsize=11)
ax.set_ylim(-15, 105); ax.set_ylabel('回答质量分数')
ax.set_title('RLHF 三阶段：每一步提升不同维度'); ax.set_xticks([])
plt.tight_layout(); plt.show()
print("预训练→SFT：续写→回答（+28分）| SFT→RLHF：回答→答得好（+24分）")

## 结论

| 问题 | 实验证据 |
|---|---|
| 奖励模型 | 实验1：好回答高分、差回答低分 |
| PPO clip | 实验2：限制每步变化，防震荡 |
| KL 惩罚 | 实验3：防偏离 SFT 太远 |
| 三阶段 | 实验4：预训练=知识、SFT=行为、RLHF=偏好 |

→ 配套阅读：`ima/第3周-Day3-RLHF人类反馈强化学习详解.md`